In [1]:
# [Colab] Google Drive 마운트 — Colab에서 실행 시 아래 주석 해제
# from google.colab import drive
# drive.mount('/content/drive')

Mounted at /content/drive


# 데이터 불러오기

In [2]:
import pandas as pd

# [VS Code]
DATA_DIR = '../data/raw'

# [Colab] Colab에서 실행 시 위 DATA_DIR 주석 처리 후 아래 두 줄 주석 해제
# from google.colab import drive; drive.mount('/content/drive')
# DATA_DIR = '/content/drive/MyDrive/why-they-leave/retail-clickstream-analysis/raw'

customers   = pd.read_csv(f'{DATA_DIR}/customers.csv')
products    = pd.read_csv(f'{DATA_DIR}/products.csv')
events      = pd.read_csv(f'{DATA_DIR}/events.csv')
sessions    = pd.read_csv(f'{DATA_DIR}/sessions.csv')
orders      = pd.read_csv(f'{DATA_DIR}/orders.csv')
order_items = pd.read_csv(f'{DATA_DIR}/order_items.csv')

# customers (고객 프로필)
- **customer_id**: Integer (null 값 없음)
- **name**: String
- **email**: String
- **country**: String
- **age**: Integer
- **signup_date**: Date, 가입일
- **marketing_opt_in**: Boolean, 마케팅 수신 동의 여부(정확하지 않음, gpt가 추론 한 거)

## null 값이 있나?
: 모든 칼럼 전부 없음.

In [3]:
customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   customer_id       20000 non-null  int64 
 1   name              20000 non-null  object
 2   email             20000 non-null  object
 3   country           20000 non-null  object
 4   age               20000 non-null  int64 
 5   signup_date       20000 non-null  object
 6   marketing_opt_in  20000 non-null  bool  
dtypes: bool(1), int64(2), object(4)
memory usage: 957.2+ KB


## customer_id는 중복된 것이 있나?
목적: 중복된 것이 있으면, 데이터 테이블  문제  

: 없음. 전부 하나씩만 존재.

In [7]:
import numpy as np
np.sum(customers['customer_id'].value_counts() > 1)

np.int64(0)

## country 별로 유저 몇명 존재?  
목적: US만 뽑아도 문제 없을 정도의 표본일까?  
결과: US가 가장 많음. 전체에서 비중이 그렇게 많지 않아서 행동 로그 데이터가 많을지를 같이 보고 판단해야할 듯.

In [11]:
country_counts = customers.groupby('country')['customer_id'].count()
country_counts.sort_values(ascending=False)

,customer_id
country,
US,3648
IN,1589
GB,1585
BR,1421
DE,1397
FR,1325
MX,1206
AU,1045
CA,1015


## Age Group별 고객 수
목적: 연령대별 고객 분포 파악  


### Age Group 별 전체 고객 수
목적: 특정 나이대에만 몰려있는지 파악하기 위함.  
결과: 10대나 70대이상 부터는 소비 페르소나라고 할 게... 정의하기 좀 애매했는데 적당한 나이대에 적당히 분포가 몰려있어 괜찮은 거 같음.

In [12]:
# Define age bins
bins = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
labels = ['0-9', '10-19', '20-29', '30-39', '40-49', '50-59', '60-69', '70-79', '80-89', '90-99']

# Create a new column 'age_group' by cutting the 'age' column into bins
customers['age_group'] = pd.cut(customers['age'], bins=bins, labels=labels, right=False)

# Count customers per age group
age_group_counts = customers.groupby('age_group')['customer_id'].count()

# Display the sorted counts
age_group_counts.sort_index()

/tmp/ipykernel_2482/3065394674.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  age_group_counts = customers.groupby('age_group')['customer_id'].count()


,customer_id
age_group,
0-9,0
10-19,682
20-29,3428
30-39,3515
40-49,3382
50-59,3453
60-69,3462
70-79,2078
80-89,0


### Age Group 별 US 고객 수   
목적: us도 마찬가지?  

결과: 골고루 많이 분포.

In [13]:
US_customers = customers[customers['country'] == 'US']

# Define age bins
bins = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
labels = ['0-9', '10-19', '20-29', '30-39', '40-49', '50-59', '60-69', '70-79', '80-89', '90-99']

# Create a new column 'age_group' by cutting the 'age' column into bins
US_customers['age_group'] = pd.cut(US_customers['age'], bins=bins, labels=labels, right=False)

# Count customers per age group
US_age_group_counts = US_customers.groupby('age_group')['customer_id'].count()

# Display the sorted counts
US_age_group_counts.sort_index()

/tmp/ipykernel_2482/1426177289.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  US_customers['age_group'] = pd.cut(US_customers['age'], bins=bins, labels=labels, right=False)
/tmp/ipykernel_2482/1426177289.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  US_age_group_counts = US_customers.groupby('age_group')['customer_id'].count()


,customer_id
age_group,
0-9,0
10-19,124
20-29,584
30-39,597
40-49,616
50-59,642
60-69,653
70-79,432
80-89,0


#products(상품) 1197  
- **product_id**: Integer
- **category**: String → null 값 없음.
- **name**: String →  null 값 없음.
- **price_usd**: Float, 상품가격
- **cost_usd**: Float, 원가
- **margin_usd**: Float, 마진

## null 값이 있나?  
결과: 없음

In [14]:
products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1197 entries, 0 to 1196
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   product_id  1197 non-null   int64  
 1   category    1197 non-null   object 
 2   name        1197 non-null   object 
 3   price_usd   1197 non-null   float64
 4   cost_usd    1197 non-null   float64
 5   margin_usd  1197 non-null   float64
dtypes: float64(3), int64(1), object(2)
memory usage: 56.2+ KB


## product_id 겹치는 거 있나  
결과: 없음.

In [15]:
np.sum(products['product_id'].value_counts() > 1)

np.int64(0)

## category 칼럼 살피기  
결과: 다 같네...

In [16]:
category_counts = products.groupby('category')['product_id'].count()
category_counts.sort_values(ascending=False)

,product_id
category,
Beauty,171
Books,171
Electronics,171
Fashion,171
Home & Kitchen,171
Sports,171
Toys,171


# sessions(세션 메타데이터) 120000 → events랑 customer 연결 고리  
- **session_id**: Integer → null 없음
- **customer_id**: Integer → null 없음
- **start_time**: Timestamp
- **device**: String
- **source**: String
- **country**: String

## null 값이 있나?  
결과: 없음

In [17]:
sessions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120000 entries, 0 to 119999
Data columns (total 6 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   session_id   120000 non-null  int64 
 1   customer_id  120000 non-null  int64 
 2   start_time   120000 non-null  object
 3   device       120000 non-null  object
 4   source       120000 non-null  object
 5   country      120000 non-null  object
dtypes: int64(2), object(4)
memory usage: 5.5+ MB


## session_id 겹치는 거 있나  
결과: 없음.

In [18]:
np.sum(sessions['session_id'].value_counts() > 1)

np.int64(0)

## 고유 customer_id 갯수?  
목적: 모든 customer에 대한 세션이 존재하나?  
결과: 전체 고객 수는 20000행, 세션이 존재하는 고객은 19945행

In [19]:
sessions['customer_id'].nunique()

19945

## 고객 행동 데이터 수집기간  
결과  
시작: 2020-01-01  
끝: 2025-10-31

In [22]:
print(sessions['start_time'].min())
print(sessions['start_time'].max())

2020-01-01T00:06:58
2025-10-31T23:34:11


## 나라별로 세션수는 얼마나 있나?  
목적: US만 필터링 해도 되는 건지 파악  

결과: US가 가장 많긴 하지만, 전체에 비해서 그렇게 큰 비중은 아님.   
근데 세션은 접속할 때만 찍히는 거니까, events table 을 뜯어봐야함.

In [23]:
session_counts = sessions.groupby('country')['session_id'].count()
session_counts.sort_values(ascending=False)

,session_id
country,
US,21913
IN,9567
GB,9451
BR,8573
DE,8332
FR,7811
MX,7286
AU,6443
CA,6057


## 유저마다 보유한 세션 수의 최댓값, 최솟값?  
목적: 최대로 많이 접속한 유저의 접속 수와 최소로 많이 접속한 유저의 접속 수 파악  
결과: 최댓값은 17, 최솟값은 1

In [26]:
session_counts = sessions.groupby('customer_id')['session_id'].count()
print(session_counts.max(),session_counts.min())

17 1


# events(행동 로그) 760958
- **event_id**: Integer
- **session_id**: Integer
- **timestamp**: Timestamp
- **event_type**: String (check_out, purchase, add_to_cart, page_view)
- **product_id**: Float → NULL 78489  
- **qty**: Float → NULL 617832
- **cart_size**: Float → check_out events에서만 값이 존재. 아마 구매할 때 장바구니에 몇개 정도 담겨있는지를 보는 거 같음.
- **payment**: String
- **discount_pct**: Float
- **amount_usd**: Float

## null 값이 있나?  
결과: product_id 외 여러 칼럼에서 결측치 보임.
다른 칼럼은 주문 관련 칼럼이라 주문하지 않은 고객에 대해선 결측 존재할 수 있음.  
하지만, product_id는 확인 필요

In [27]:
events.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 760958 entries, 0 to 760957
Data columns (total 10 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   event_id      760958 non-null  int64  
 1   session_id    760958 non-null  int64  
 2   timestamp     760958 non-null  object 
 3   event_type    760958 non-null  object 
 4   product_id    682469 non-null  float64
 5   qty           143126 non-null  float64
 6   cart_size     44909 non-null   float64
 7   payment       33580 non-null   object 
 8   discount_pct  33580 non-null   float64
 9   amount_usd    33580 non-null   float64
dtypes: float64(5), int64(2), object(3)
memory usage: 58.1+ MB


## event_id 겹치는 거 있나  
결과: 없음.

In [29]:
np.sum(events['event_id'].value_counts() > 1)

np.int64(0)

## session_id 겹치는 거 있나  
결과: 없음.

In [28]:
np.sum(events['event_id'].value_counts() > 1)

np.int64(0)

## event_type 칼럼 살피기  
결과: 조회가 가장 많음, 행동은 네가지

In [30]:
event_type_counts = events.groupby('event_type')['event_id'].count()
event_type_counts.sort_values(ascending=False)

,event_id
event_type,
page_view,539343
add_to_cart,143126
checkout,44909
purchase,33580


## product_id null 값 살피기  
목적: 행동 로그 분석할 때 product_id 없으면 곤란  
결과: 구매와 관련된 checkout이랑 purchase에 있음.   
그리고 checkout과 purchase event는 모두 product_id null 값  

44909(check out event 수) + 33580(purchase event 수) = 78489(product_id 결측치)

In [31]:
events_product_id_null = events[events['product_id'].isnull()]
null_product_id_event_type_counts = events_product_id_null.groupby('event_type')['event_id'].count()
null_product_id_event_type_counts.sort_values(ascending=False)

,event_id
event_type,
checkout,44909
purchase,33580


# orders(주문) → 상품명 알아내려면 order_items랑 조인해야함.  
전체 → 33580, US → 6138

- **order_id**: Integer
- **customer_id**: Integer → null 값 없음
- **order_time**: Timestamp
- **payment_method**: String

paypal, wallet, cod, card

- **discount_pct**: Integer → 할인율
- **subtotal_usd**: Float → 할인 전 전체 금액
- **total_usd**: Float → 할인 받은 후 지불한 금액
- **country**: String
- **device**: String

## null 값이 있나?  
결과: 없음

In [32]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33580 entries, 0 to 33579
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   order_id        33580 non-null  int64  
 1   customer_id     33580 non-null  int64  
 2   order_time      33580 non-null  object 
 3   payment_method  33580 non-null  object 
 4   discount_pct    33580 non-null  int64  
 5   subtotal_usd    33580 non-null  float64
 6   total_usd       33580 non-null  float64
 7   country         33580 non-null  object 
 8   device          33580 non-null  object 
 9   source          33580 non-null  object 
dtypes: float64(2), int64(3), object(5)
memory usage: 2.6+ MB


## order_id 겹치는 거 있나  
결과: 없음.

In [33]:
np.sum(orders['order_id'].value_counts() > 1)

np.int64(0)

## 고유 customer_id 수?  
목적: 고객 중에 주문한 적이 있는 customer 수는 몇명정도  
결과: 16268 (전체 고객 중 20000)

In [34]:
orders['customer_id'].nunique()

16268

## payment_method 칼럼 살피기  
결과: 카드 페이팔 월렛 코드가 있고 카드가 가장 많음.

In [37]:
payment_counts = orders.groupby('payment_method')['order_id'].count()
payment_counts.sort_values(ascending=False)

,order_id
payment_method,
card,23455
paypal,5032
wallet,3373
cod,1720


## device 칼럼 살피기

In [38]:
device_counts = orders.groupby('device')['order_id'].count()
device_counts.sort_values(ascending=False)

,order_id
device,
mobile,18469
desktop,12750
tablet,2361


## source 칼럼 살피기

In [39]:
source_counts = orders.groupby('source')['order_id'].count()
source_counts.sort_values(ascending=False)

,order_id
source,
organic,11268
direct,8387
paid,4121
social,4024
email,3056
referral,2724


# order_items(주문 상세 정보)  
- **order_id**: Integer → 결측 없음
- **product_id**: Integer → 결측 없음
- **unit_price_usd**: Float → 고유 상품 하나에 대한 단일 가격
- **quantity**: Integer → 주문 한 건당 동일한 아이템 품목의 수량
- **line_total_usd**: Float → unit_price_usd * quantity → 한 주문 내 동일 품목에 대한 총 지출 금액  

→ 한번 주문할 때마다 동일한 order_id가 찍히고

→ 한번 주문할 때 다른 상품들을 주문하게 되면 상품 id 갯수만큼 행수가 찍힘


## null 값이 있나?  
결과: 없음

In [41]:
order_items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 59163 entries, 0 to 59162
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   order_id        59163 non-null  int64  
 1   product_id      59163 non-null  int64  
 2   unit_price_usd  59163 non-null  float64
 3   quantity        59163 non-null  int64  
 4   line_total_usd  59163 non-null  float64
dtypes: float64(2), int64(3)
memory usage: 2.3 MB
